<a href="https://colab.research.google.com/github/joyhwp/lg_multimodal_cej_analysis/blob/main/01_llm_cej_mapping_initial_approach.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
df = pd.read_csv('/kaggle/input/datasets/joycep7/product6/lg_reviews_M876GBB231.csv')
df

In [ ]:
keep_cols = [
    'review_id',
    'model_name_code',
    'product_name',
    'rating',
    'written_date',
    'review_text',
    'product_option',
    'image_paths'
]

df = df[keep_cols]
df

In [ ]:
# =========================
# 3. 전체 리뷰 분석용 데이터 준비
# =========================

df = df.copy()

df["review_text_clean"] = (
    df["review_text"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["review_text_len"] = df["review_text_clean"].str.len()

# 원본 순서 기준 review_order 부여
df["review_order"] = range(1, len(df) + 1)

def extract_option_value(option_text, key):
    if not isinstance(option_text, str):
        return None

    parts = option_text.split("|")

    for part in parts:
        part = part.strip()
        if part.startswith(key + ":"):
            return part.replace(key + ":", "").strip()

    return None

df["material_option"] = df["product_option"].apply(
    lambda x: extract_option_value(x, "소재")
)

df["color_option"] = df["product_option"].apply(
    lambda x: extract_option_value(x, "색상")
)

# 이번 분석은 필터링 없이 전체 리뷰 사용
df_sample = df.copy()

print("전체 리뷰 수:", len(df))
print("분석 대상 리뷰 수:", len(df_sample))

print("\n리뷰 글자수 통계:")
display(df_sample["review_text_len"].describe())

print("\nmaterial_option 분포:")
display(df_sample["material_option"].value_counts(dropna=False))

print("\ncolor_option 분포:")
display(df_sample["color_option"].value_counts(dropna=False))

display(
    df_sample[
        [
            "review_order",
            "review_id",
            "rating",
            "written_date",
            "product_option",
            "material_option",
            "color_option",
            "review_text_clean",
            "review_text_len"
        ]
    ].head(10)
)

In [ ]:
def build_text_prompt_cej_only(row):
    review_text = str(row["review_text_clean"])

    prompt = f"""
당신은 LG 냉장고 리뷰 분석가입니다.

목표:
리뷰 텍스트를 CEJ 단계별로 나누고, 각 단계의 근거를 추출하세요.
같은 CEJ 단계는 반드시 하나의 행으로 합치세요.
cej_rows의 최대 행 수는 3개이며, 각 행의 cej_stage는 "구매", "설치", "사용" 중 중복 없이 하나씩만 사용하세요.

핵심 규칙:
- 리뷰에 실제로 언급된 CEJ 단계만 cej_rows에 넣으세요.
- evidence는 반드시 리뷰 원문에서 그대로 복사한 짧은 표현만 쓰세요. 맞춤법/오타 수정 금지.
- 추론하지 마세요. 텍스트에 보이는 것만 추출하세요.
- JSON만 출력하세요. 마크다운 코드블록 쓰지 마세요.

[리뷰 텍스트]
{review_text}

CEJ 단계 정의:
- 구매: 가격, 할인, 환급, 혜택, 구매결정, 구매계기, 새로 구매, 기존 제품 교체, 이사/혼수, 포장, AS 기대감
- 설치: 배송, 설치과정, 설치기사, 반입, 공간배치, 냉장고장에 맞춤, 빌트인 배치, 초기세팅, 사용법 안내
- 사용: 일상사용, 기능작동, 편의성, 소음, 수납, 용량 체감, 냉장/냉동 사용, 유지관리, 청소, AS경험, 잔고장/고장 관련 경험, 성능 일반 평가

반환 JSON:
{{
  "review_id": "{row["review_id"]}",
  "model_name_code": "{row["model_name_code"]}",
  "review_order": {int(row["review_order"])},
  "cej_rows": [
    {{
      "cej_stage": "구매",
      "stage_evidence": "리뷰 원문에서 해당 CEJ 단계 근거를 그대로 복사"
    }}
  ]
}}

작성 규칙:
- cej_stage는 "구매", "설치", "사용" 중 하나만 쓰세요.
- cej_rows에는 실제 언급된 단계만 넣으세요.
- 모든 stage_evidence가 실제 [리뷰 텍스트] 안에 포함되어 있는지 확인하세요.
"""
    return prompt

In [ ]:
# =========================
# 5. 모델 설치
# =========================

!pip install -U transformers accelerate bitsandbytes -q

In [ ]:
# =========================
# 6. 모델 선택
# =========================

import torch

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

print("사용 모델:", MODEL_ID)
print("CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
pip install -U bitsandbytes>=0.46.1

In [ ]:
# =========================
# 7. 모델 로딩
# =========================

import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
    trust_remote_code=True
)

model.eval()

print("모델 로딩 완료:", MODEL_ID)

In [ ]:
# =========================
# 8. JSON 추출 함수
# =========================

import json
import re

def extract_json_from_text(text):
    if text is None:
        return None

    text = text.strip()

    text = re.sub(r"^```json", "", text.strip(), flags=re.IGNORECASE)
    text = re.sub(r"^```", "", text.strip())
    text = re.sub(r"```$", "", text.strip())

    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1 or end <= start:
        return None

    json_str = text[start:end+1]

    try:
        return json.loads(json_str)
    except Exception:
        return None

In [ ]:
# =========================
# 9. CEJ 결과 후처리 함수 (CEJ only)
# =========================

def clean_cej_parsed_json(parsed):
    if parsed is None:
        return None

    cej_rows = parsed.get("cej_rows") or []
    merged = {}  # stage → row 딕셔너리로 중복 병합

    for cej in cej_rows:
        if not isinstance(cej, dict):
            continue

        stage = cej.get("cej_stage")
        evidence = cej.get("stage_evidence")

        if stage not in ["구매", "설치", "사용"]:
            continue

        if evidence is None or str(evidence).strip() == "":
            continue

        evidence = evidence.strip()

        if stage not in merged:
            merged[stage] = {
                "cej_stage":      stage,

                "stage_evidence": evidence
            }
        else:
            # 같은 단계 중복 → evidence 이어붙이기, sentiment는 다르면 positive 우선
            prev = merged[stage]
            if evidence not in prev["stage_evidence"]:
                prev["stage_evidence"] = prev["stage_evidence"] + " , " + evidence


    # 구매→설치→사용 순서 정렬
    order = ["구매", "설치", "사용"]
    parsed["cej_rows"] = [merged[s] for s in order if s in merged]
    return parsed

In [ ]:
# =========================
# 10. 텍스트 분석 함수
# =========================

def analyze_text_only(row, max_new_tokens=1000):
    prompt = build_text_prompt_cej_only(row)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    try:
        try:
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False
            )
        except TypeError:
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

        inputs = tokenizer(
            text,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=True,
                eos_token_id=tokenizer.eos_token_id
            )

        input_len = inputs["input_ids"].shape[-1]
        generated_ids_trimmed = generated_ids[:, input_len:]

        output_text = tokenizer.decode(
            generated_ids_trimmed[0],
            skip_special_tokens=True
        )

        parsed_json = extract_json_from_text(output_text)
        parsed_json = clean_cej_parsed_json(parsed_json)

        return {
            "success": parsed_json is not None,
            "review_order": int(row["review_order"]),
            "review_id": str(row["review_id"]),
            "raw_response": output_text,
            "parsed_json": parsed_json,
            "error": None
        }

    except Exception as e:
        return {
            "success": False,
            "review_order": int(row["review_order"]),
            "review_id": str(row["review_id"]),
            "raw_response": None,
            "parsed_json": None,
            "error": str(e)
        }

In [ ]:
# 11. CEJ-row 결과 flatten 함수 (CEJ only)
# =========================

def flatten_cej_result(row, text_result):
    parsed = text_result.get("parsed_json")

    base = {
        "review_id":        str(row["review_id"]),
        "model_name_code":  row.get("model_name_code"),
        "rating":           int(row["rating"]) if pd.notna(row.get("rating")) else None,
        "written_date":     row.get("written_date"),
        "product_option":   row.get("product_option"),
        "material_option":  row.get("material_option"),
        "color_option":     row.get("color_option"),
        "review_text":      row.get("review_text_clean"),
    }

    null_cej = {
        "cej_stage":     None,

        "cej_evidence":  None,
    }

    # 파싱 실패
    if parsed is None:
        fail_row = base.copy()
        fail_row.update(null_cej)
        return [fail_row]

    cej_rows = parsed.get("cej_rows") or []

    # cej_rows 비어있음
    if len(cej_rows) == 0:
        empty_row = base.copy()
        empty_row.update(null_cej)
        return [empty_row]

    # 정상
    flat_rows = []
    for cej in cej_rows:
        out = base.copy()
        out.update({
            "cej_stage":     cej.get("cej_stage"),

            "cej_evidence":  cej.get("stage_evidence"),
        })
        flat_rows.append(out)

    return flat_rows

In [ ]:
# =========================
# 14. 전체 실행 저장 경로 설정 (캐글)
# =========================
from pathlib import Path
import time

output_dir = Path("/kaggle/working/cej_output")
output_dir.mkdir(parents=True, exist_ok=True)

output_json       = output_dir / "cej_results_kaggle.json"
output_csv        = output_dir / "cej_results_kaggle.csv"
output_jsonl      = output_dir / "cej_results_kaggle.jsonl"
progress_log_path = output_dir / "progress_kaggle.log"

print("저장 경로:", output_dir)

In [ ]:
# =========================
# 15. 저장 / 이어서 실행 함수
# =========================

def append_progress_log(message):
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
    with open(progress_log_path, "a", encoding="utf-8") as f:
        f.write(f"[{timestamp}] {message}\n")


def load_saved_jsonl(jsonl_path):
    results = []

    if jsonl_path.exists():
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    results.append(json.loads(line))
                except:
                    pass

    return results


def save_outputs(results):
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    all_flat_rows = []

    row_map = {
        str(r["review_id"]): r
        for _, r in df_sample.iterrows()
    }

    for result in results:
        review_id = str(result.get("review_id"))
        row = row_map.get(review_id)

        if row is None:
            continue

        flat_rows = flatten_cej_result(row, result)
        all_flat_rows.extend(flat_rows)  # ← elapsed_sec 제거

    df_out = pd.DataFrame(all_flat_rows)
    df_out.to_csv(output_csv, index=False, encoding="utf-8-sig")

    return df_out


In [ ]:
# =========================
# 16. 전체 실행
# =========================

saved_results = load_saved_jsonl(output_jsonl)

done_review_ids = {
    str(r.get("review_id"))
    for r in saved_results
}

print("이미 저장된 결과 수:", len(saved_results))
print("이미 완료된 review_id 수:", len(done_review_ids))

if len(saved_results) > 0:
    df_saved = save_outputs(saved_results)
    print("기존 결과 기반 CSV 갱신 완료:", output_csv)
    print("현재 CSV rows:", len(df_saved))

all_results = saved_results.copy()

start_all = time.time()

for _, row in df_sample.iterrows():
    review_id = str(row["review_id"])

    if review_id in done_review_ids:
        print(f"skip review_order={row['review_order']} review_id={review_id}")
        continue

    print("=" * 100)
    print("review_order:", row["review_order"])
    print("review_id:", review_id)
    print("review_text_len:", row["review_text_len"])

    append_progress_log(
        f"START review_order={row['review_order']} review_id={review_id}"
    )

    try:
        start = time.time()

        text_result = analyze_text_only(
            row,
            max_new_tokens=1000
        )

        elapsed = time.time() - start

        text_result["elapsed_sec"] = round(elapsed, 2)

    except Exception as e:
        err = traceback.format_exc()

        text_result = {
            "success": False,
            "review_order": int(row["review_order"]),
            "review_id": review_id,
            "raw_response": None,
            "parsed_json": None,
            "error": str(e),
            "elapsed_sec": None,
            "traceback": err
        }

        print("ERROR:", str(e))
        append_progress_log(f"ERROR review_id={review_id} error={str(e)}")

    # JSONL 즉시 저장
    with open(output_jsonl, "a", encoding="utf-8") as f:
        f.write(json.dumps(text_result, ensure_ascii=False) + "\n")

    all_results.append(text_result)
    done_review_ids.add(review_id)

    # 매 리뷰마다 JSON / CSV 갱신
    df_out = save_outputs(all_results)

    print("저장 완료:", review_id)
    print("success:", text_result.get("success"))
    print("elapsed_sec:", text_result.get("elapsed_sec"))
    print("현재 결과 수:", len(all_results))
    print("현재 CSV rows:", len(df_out))

    append_progress_log(
        f"DONE review_order={row['review_order']} review_id={review_id} "
        f"success={text_result.get('success')} elapsed={text_result.get('elapsed_sec')}"
    )

total_elapsed = time.time() - start_all

print("=" * 100)
print("전체 실행 완료")
print("총 결과 수:", len(all_results))
print("총 소요 시간:", round(total_elapsed / 60, 2), "분")
print("JSONL:", output_jsonl)
print("JSON:", output_json)
print("CSV:", output_csv)

append_progress_log(
    f"ALL_DONE total_results={len(all_results)} total_elapsed_min={round(total_elapsed / 60, 2)}"
)